## **Handling Large Dataset**

In [ ]:
# Use Datasets library to handle large datasets(For CPU)
from datasets import load_dataset, Dataset
import psutil

def load_and_process_large_dataset(dataset_name, num_proc):
    # Load the dataset
    dataset = load_dataset(dataset_name, streaming=True)
    # Define a preprocessing function
    def preprocess_function(example):
        # Implement preprocess logic here
        return example
    # Apply preprocess in parallel
    processed_dataset = dataset.map(
        preprocess_function,
        batched=True,
        num_proc=num_proc,
        remove_columns=dataset['train'].column_names
    )
    return processed_dataset

# Determine the number of cpu cores for parallel processing
num_cores = psutil.cpu_count(logical=False)
# Load and process the large dataset(e.g. c4 dataset)
large_dataset = load_and_process_large_dataset("c4", num_proc=num_cores)
# Print the first few examples
for example in large_dataset["train"].take(5):
    print(example)

In [ ]:
# Use datasets library to handle large datasets(For GPU)
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

def load_and_process_dataset(dataset_name, batch_size):
    dataset = load_dataset(dataset_name, streaming=True)
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    def preprocess(examples):
        return tokenizer(
            examples["text"], padding="max_length",
            truncation=True, return_tensors="pt"
        )
    def process_batch(batch):
        return {k: v.to(device) for k, v in preprocess(batch).items()}
    return DataLoader(
        dataset["train"].map(process_batch),
        batch_size=batch_size, num_workers=2,
        pin_memory=True
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataloader = load_and_process_dataset("c4", batch_size=32)
for i, batch in enumerate(dataloader):
    if i > 5: break
    print(f"Batch {i}:", {k: v.shape for k, v in batch.items()})